# 04 - Cleaning and Dataset Validation

Strict dataset cleaning, global validation, and safe price estimation before backend integration.


In [ ]:
import json
import re
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data/processed")
SOURCE_FILES = {
    "wisata": DATA_DIR / "wisata_final.csv",
    "kuliner": DATA_DIR / "kuliner_final.csv",
    "hotel": DATA_DIR / "hotel_final.csv",
}

REQUIRED_COLUMNS = ["name", "category", "rating", "address", "subtypes"]
OPTIONAL_PRICE_COLUMNS = ["price", "price_level", "range", "prices"]
FINAL_COLUMNS = ["name", "category", "rating", "address", "subtypes", "type", "price_estimate"]

SAFE_PRICE_MIN = 10_000
SAFE_PRICE_MAX = 1_000_000
PRICE_LEVEL_MAP = {1: 25_000, 2: 50_000, 3: 80_000, 4: 150_000, 5: 300_000}
CURRENCY_TO_IDR = {
    "rp": 1,
    "idr": 1,
    "€": 17_500,
    "eur": 17_500,
    "$": 16_000,
    "usd": 16_000,
}

raw_frames = {}
for dataset_type, path in SOURCE_FILES.items():
    frame = pd.read_csv(path)
    frame.columns = frame.columns.str.strip().str.lower()
    raw_frames[dataset_type] = frame
    existing_price_columns = [column for column in OPTIONAL_PRICE_COLUMNS if column in frame.columns]
    print(dataset_type, frame.shape, "price columns:", existing_price_columns)


In [ ]:
def normalize_text(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).strip())


def normalize_category(value):
    text = normalize_text(value).lower()
    return text if text else "unknown"


def safe_numeric_series(values, default_value):
    numeric = pd.to_numeric(pd.Series(values), errors="coerce")
    numeric = numeric.fillna(default_value)
    numeric = numeric.clip(lower=SAFE_PRICE_MIN, upper=SAFE_PRICE_MAX)
    return numeric.astype("int64")


def parse_price_string(value):
    """Extract only explicit currency/price-level values; ignore URLs, dates, IDs, and star ratings."""
    try:
        if pd.isna(value):
            return []
    except (TypeError, ValueError):
        pass

    text = str(value).strip().lower()
    if not text or text in {"nan", "none", "null", "-"}:
        return []

    if re.search(r"\bstars?\b", text):
        return []

    dollar_level = re.fullmatch(r"\$+", text)
    if dollar_level:
        return [PRICE_LEVEL_MAP.get(len(text), 300_000)]

    price_level_match = re.search(r"(?:price[_ ]?level|level)\D*(\d)", text)
    if price_level_match:
        return [PRICE_LEVEL_MAP.get(int(price_level_match.group(1)), 300_000)]

    candidates = []
    currency_pattern = r"(rp|idr|€|eur|\$|usd)\s*([0-9]+(?:[.,][0-9]+)*)"
    for currency, amount_text in re.findall(currency_pattern, text, flags=re.IGNORECASE):
        currency = currency.lower()
        amount = pd.to_numeric(amount_text.replace(",", "."), errors="coerce")
        if pd.isna(amount):
            continue
        amount = float(amount)
        if currency in {"rp", "idr"} and amount < 1000:
            amount *= 1000
        candidates.append(amount * CURRENCY_TO_IDR[currency])

    # Indonesian shorthand such as "50k" or "80 rb" is acceptable because it is explicitly price-like.
    for amount_text in re.findall(r"\b([0-9]+(?:[.,][0-9]+)?)\s*(?:k|rb|ribu)\b", text):
        amount = pd.to_numeric(amount_text.replace(",", "."), errors="coerce")
        if not pd.isna(amount):
            candidates.append(float(amount) * 1000)

    return candidates


def extract_prices_from_json(value):
    try:
        parsed = json.loads(value) if isinstance(value, str) else value
    except (TypeError, ValueError, json.JSONDecodeError):
        return []

    candidates = []
    price_keys = {"price", "price_total", "amount", "value"}

    def walk(item):
        if isinstance(item, dict):
            for key, val in item.items():
                if str(key).lower() in price_keys:
                    candidates.extend(parse_price_string(val))
                elif isinstance(val, (dict, list)):
                    walk(val)
        elif isinstance(item, list):
            for child in item:
                walk(child)

    walk(parsed)
    return candidates


def parse_numeric_price(value):
    """Safely return a clipped rupiah estimate or pd.NA. Never parses arbitrary URL numbers."""
    try:
        json_candidates = extract_prices_from_json(value)
        text_candidates = parse_price_string(value)
        candidates = json_candidates or text_candidates
        if not candidates:
            return pd.NA

        numeric = safe_numeric_series(candidates, default_value=pd.NA)
        numeric = numeric[(numeric >= SAFE_PRICE_MIN) & (numeric <= SAFE_PRICE_MAX)]
        if numeric.empty:
            return pd.NA
        return int(numeric.median())
    except (OverflowError, ValueError, TypeError):
        return pd.NA


def rule_based_price(row):
    profile = f"{row.get('name', '')} {row.get('category', '')} {row.get('subtypes', '')}".lower()
    dataset_type = row["type"]

    if dataset_type == "wisata":
        if any(keyword in profile for keyword in ["pantai", "beach"]):
            return 10_000
        if any(keyword in profile for keyword in ["candi", "temple"]):
            return 50_000
        if "museum" in profile:
            return 25_000
        return 25_000

    if dataset_type == "kuliner":
        if any(keyword in profile for keyword in ["cafe", "coffee", "kopi"]):
            return 50_000
        if any(keyword in profile for keyword in ["restoran", "restaurant"]):
            return 80_000
        return 50_000

    if dataset_type == "hotel":
        return 300_000

    raise ValueError(f"Unknown type: {dataset_type}")


def build_price_estimate(row):
    for column in OPTIONAL_PRICE_COLUMNS:
        if column in row.index:
            parsed = parse_numeric_price(row[column])
            if not pd.isna(parsed):
                return parsed
    return rule_based_price(row)


In [ ]:
clean_frames = []
validation_rows = []

for dataset_type, frame in raw_frames.items():
    missing_required = sorted(set(REQUIRED_COLUMNS) - set(frame.columns))
    if missing_required:
        raise ValueError(f"{dataset_type} is missing required columns: {missing_required}")

    working = frame.copy()
    for column in OPTIONAL_PRICE_COLUMNS:
        if column not in working.columns:
            working[column] = pd.NA

    working = working[REQUIRED_COLUMNS + OPTIONAL_PRICE_COLUMNS].copy()
    working["name"] = working["name"].apply(normalize_text)
    working["address"] = working["address"].apply(normalize_text)
    working["subtypes"] = working["subtypes"].apply(normalize_text).replace("", "unknown")
    working["category"] = working["category"].apply(normalize_category)
    working["rating"] = pd.to_numeric(working["rating"], errors="coerce")
    working["type"] = dataset_type

    before_drop = len(working)
    working = working[(working["name"] != "") & (working["address"] != "")].copy()
    dropped_critical = before_drop - len(working)

    before_dedup = len(working)
    working = working.drop_duplicates(subset=["name", "address"], keep="first").copy()
    duplicates_removed = before_dedup - len(working)

    raw_price_estimates = working.apply(build_price_estimate, axis=1)
    working["price_estimate"] = safe_numeric_series(raw_price_estimates, default_value=rule_based_price({"type": dataset_type}))

    if working["price_estimate"].isna().any():
        raise AssertionError(f"{dataset_type}: price_estimate contains nulls")
    if (working["price_estimate"] < SAFE_PRICE_MIN).any() or (working["price_estimate"] > SAFE_PRICE_MAX).any():
        raise AssertionError(f"{dataset_type}: unsafe price_estimate found")

    validation_rows.append({
        "type": dataset_type,
        "raw_rows": len(frame),
        "dropped_missing_name_or_address": dropped_critical,
        "duplicates_removed_within_type": duplicates_removed,
        "rows_after_cleaning": len(working),
        "subtypes_unknown": int((working["subtypes"] == "unknown").sum()),
        "price_estimate_min": int(working["price_estimate"].min()),
        "price_estimate_max": int(working["price_estimate"].max()),
    })

    clean_frames.append(working[FINAL_COLUMNS])

validation_report = pd.DataFrame(validation_rows)
validation_report


In [ ]:
df_all = pd.concat(clean_frames, ignore_index=True)
global_duplicates_removed = int(df_all.duplicated(subset=["name", "address"]).sum())
df_all = df_all.drop_duplicates(subset=["name", "address"], keep="first").reset_index(drop=True)

expected_types = {"wisata", "kuliner", "hotel"}
actual_types = set(df_all["type"].unique())
if actual_types != expected_types:
    raise AssertionError(f"Type mismatch. Expected {expected_types}, got {actual_types}")

critical_columns = ["name", "category", "address", "subtypes", "type", "price_estimate"]
if df_all[critical_columns].isna().sum().sum() != 0:
    raise AssertionError("Nulls remain in critical columns.")

if df_all.duplicated(subset=["name", "address"]).any():
    raise AssertionError("Duplicate places still exist by name + address.")

if (df_all["price_estimate"] < SAFE_PRICE_MIN).any() or (df_all["price_estimate"] > SAFE_PRICE_MAX).any():
    raise AssertionError("price_estimate outside safe range.")

if not pd.api.types.is_integer_dtype(df_all["price_estimate"]):
    df_all["price_estimate"] = pd.to_numeric(df_all["price_estimate"], errors="coerce").fillna(300_000).clip(SAFE_PRICE_MIN, SAFE_PRICE_MAX).astype("int64")

print("Dataset shape:", df_all.shape)
print("Null count:", df_all.isna().sum().to_dict())
print("Duplicate count:", int(df_all.duplicated(subset=["name", "address"]).sum()))
print("Global duplicates removed:", global_duplicates_removed)
print("Type counts:", df_all["type"].value_counts().to_dict())
print("price_estimate min/max:", int(df_all["price_estimate"].min()), int(df_all["price_estimate"].max()))
print(df_all.groupby("type")["price_estimate"].describe())

df_all.head()


In [ ]:
output_path = DATA_DIR / "merged_dataset.csv"
df_all.to_csv(output_path, index=False)
print(f"Saved clean dataset to {output_path}")
